In [ ]:
#| default_exp renderers.seadreem

# renderers.seadreem

> Renderer for the Seadreem model via Replicate.
>
> Supports LoRA and negative prompts. No reference image input.
> API key: `REPLICATE_API_TOKEN` environment variable.
> Set `renderer.seadreem.model_ref` in your config to the Replicate model string.

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations
import os
from pathlib import Path

from manhualizer.config import OutputConfig, RendererConfig
from manhualizer.models import Panel, RenderResult
from manhualizer.render import BaseRenderer, ModelSpec

In [ ]:
#| export
class SeadreemRenderer(BaseRenderer):
    """Image generation via the Seadreem model on Replicate.

    Supports LoRA weights and negative prompts.
    No reference image input — character consistency via text prompts only.

    Requires: REPLICATE_API_TOKEN environment variable.
    Config: renderer.seadreem.model_ref (Replicate model string)
    """

    def __init__(self, model_spec: ModelSpec, config: RendererConfig):
        super().__init__(model_spec, config)
        self._model_cfg = config.seadreem

    def _client(self):
        import replicate  # type: ignore
        if not os.environ.get("REPLICATE_API_TOKEN"):
            raise EnvironmentError("REPLICATE_API_TOKEN is not set")
        return replicate

    def _build_input(self, panel: Panel, output_cfg: OutputConfig) -> dict:
        w, h = output_cfg.resolved_dimensions()
        inp: dict = {
            "prompt": panel.visual_prompt,
            "width": w,
            "height": h,
            "num_outputs": 1,
        }
        # Negative prompt
        from manhualizer.prompts import load_templates
        # Use a minimal negative prompt from config if available
        # (proper template is injected by the pipeline; this is a fallback)
        if self.config.loras:
            # Pass LoRA as extra_loras if the model supports it
            inp["extra_loras"] = [
                {"url": lora.path, "scale": lora.strength}
                for lora in self.config.loras
            ]
        return inp

    async def render_async(
        self,
        panel: Panel,
        output_dir: Path,
        output_cfg: OutputConfig,
        reference_images: dict[str, Path] | None = None,
    ) -> RenderResult:
        import asyncio
        return await asyncio.get_event_loop().run_in_executor(
            None, self._render_sync, panel, output_dir, output_cfg
        )

    def _render_sync(self, panel: Panel, output_dir: Path, output_cfg: OutputConfig) -> RenderResult:
        import httpx
        replicate = self._client()

        if not self._model_cfg.model_ref:
            raise ValueError(
                "renderer.seadreem.model_ref is not set. "
                "Set it to your Replicate model string (e.g. 'owner/model:version')."
            )

        inp = self._build_input(panel, output_cfg)
        output = replicate.run(self._model_cfg.model_ref, input=inp)

        # Replicate returns a list of URLs or file-like objects
        url = output[0] if isinstance(output[0], str) else output[0].url
        image_bytes = httpx.get(url).content

        out_path = output_dir / f"panel_{panel.panel_number:04d}.{output_cfg.format}"
        out_path.write_bytes(image_bytes)

        return RenderResult(
            panel_number=panel.panel_number,
            image_path=out_path,
            backend_used=self.model_spec.name,
            prompt_used=panel.visual_prompt,
            metadata={"model_ref": self._model_cfg.model_ref, "loras": len(self.config.loras)},
        )

In [ ]:
# Construction test (no API call)
from manhualizer.render import MODELS
from manhualizer.config import RendererConfig
from manhualizer.renderers.seadreem import SeadreemRenderer

renderer = SeadreemRenderer(MODELS["seadreem"], RendererConfig())
assert renderer.model_spec.capabilities.lora
assert renderer.model_spec.capabilities.negative_prompt
assert not renderer.model_spec.capabilities.reference_images
print("SeadreemRenderer OK")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()